# Cargar datos - CustomerChurnX

Este notebook corresponde al avance V1.0.1 del PI M5. Carga la base `Base_de_datos.csv`, valida la estructura esperada y deja documentadas reglas m?nimas de calidad para el pipeline.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path.cwd() / "Base_de_datos.csv",
    Path.cwd().parent / "Base_de_datos.csv",
    Path.cwd() / "mlops_pipeline" / "Base_de_datos.csv",
]
DATA_PATH = next((path for path in candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontro Base_de_datos.csv desde el directorio actual")
ROOT = DATA_PATH.parent
DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
df.head()

## Validaci?n de esquema

La base debe contener variables de perfil, comportamiento, pagos y la variable objetivo `churn`.

In [ ]:
expected_columns = [
    "customer_id", "signup_month", "age", "tenure_months", "region", "channel",
    "plan", "sessions_week", "avg_session_min", "notif_click_rate",
    "support_tickets_3m", "discount_pct_3m", "late_payments_6m",
    "auto_renew", "churn"
]
missing = sorted(set(expected_columns) - set(df.columns))
extra = sorted(set(df.columns) - set(expected_columns))
print("Columnas faltantes:", missing)
print("Columnas adicionales:", extra)
assert not missing, "La base no cumple el esquema m?nimo requerido"

In [ ]:
df.info()

## Revisi?n inicial de calidad

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "nulos_pct": (df.isna().mean() * 100).round(2),
    "unicos": df.nunique()
})
quality

In [ ]:
duplicate_customers = df["customer_id"].duplicated().sum()
print("Clientes duplicados:", duplicate_customers)
print("Valores objetivo:")
print(df["churn"].value_counts(normalize=True).rename("proporcion"))

## Reglas de validaci?n propuestas

- `customer_id` debe ser ?nico.
- `churn` y `auto_renew` deben ser binarios.
- Las tasas `notif_click_rate` y `discount_pct_3m` deben estar entre 0 y 1.
- Variables de conteo como sesiones, tickets, mora y antig?edad no deben ser negativas.
- Variables categ?ricas (`region`, `channel`, `plan`) deben tratarse como texto y codificarse en el pipeline.

In [ ]:
validation_results = {
    "customer_id_unico": df["customer_id"].is_unique,
    "churn_binario": set(df["churn"].dropna().unique()).issubset({0, 1}),
    "auto_renew_binario": set(df["auto_renew"].dropna().unique()).issubset({0, 1}),
    "notif_click_rate_0_1": df["notif_click_rate"].between(0, 1).all(),
    "discount_pct_3m_0_1": df["discount_pct_3m"].between(0, 1).all(),
    "sin_conteos_negativos": (df[["tenure_months", "sessions_week", "support_tickets_3m", "late_payments_6m"]] >= 0).all().all(),
}
pd.Series(validation_results, name="cumple")

## Dataset listo para EDA

La base queda cargada y validada para continuar con comprensi?n exploratoria, ingenier?a de caracter?sticas, modelado y monitoreo.